# FarmGemma Training on Google Colab

**Runtime**: T4 GPU (16GB VRAM) - FREE

## Training Data (from HuggingFace):
- KisanVaani/agriculture-qa-english-only: 22.6K samples
- Mahesh2841/Agriculture: 5.9K samples
- Dharine/agriculture-10k: 10K samples
- YuvrajSingh9886/Agriculture-Plan-Diseases-QA-Pairs-Dataset: 2.6K samples

**Total: ~41,000 samples**

In [ ]:
# Install dependencies
!pip install -q transformers datasets peft accelerate bitsandbytes sentencepiece protobuf
!pip install -q torch torchvision torchaudio
!pip install -q huggingface_hub
print("Dependencies installed!")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
MODEL_DIR = "/content/drive/MyDrive/farmgemma"
import os
os.makedirs(MODEL_DIR, exist_ok=True)
print(f"Model will save to {MODEL_DIR}")

In [ ]:
# Login to HuggingFace
from huggingface_hub import login
token = input("Enter HuggingFace token: ")
login(token=token)
print("Logged in!")

## LOAD HUGGINGFACE DATASETS

In [ ]:
# Load HuggingFace datasets
from datasets import load_dataset, concatenate_datasets
print("Loading datasets...")

datasets = {}

try:
    ds = load_dataset("KisanVaani/agriculture-qa-english-only", split="train")
    datasets["AgriQA"] = ds
    print(f"AgriQA: {len(ds)} samples")
    print(f"   Columns: {ds.column_names}")
except Exception as e:
    print(f"AgriQA failed: {e}")

try:
    ds = load_dataset("Mahesh2841/Agriculture", split="train")
    datasets["Agriculture"] = ds
    print(f"Agriculture: {len(ds)} samples")
    print(f"   Columns: {ds.column_names}")
except Exception as e:
    print(f"Agriculture failed: {e}")

try:
    ds = load_dataset("Dharine/agriculture-10k", split="train")
    datasets["Agriculture-10k"] = ds
    print(f"Agriculture-10k: {len(ds)} samples")
    print(f"   Columns: {ds.column_names}")
except Exception as e:
    print(f"Agriculture-10k failed: {e}")

try:
    ds = load_dataset("YuvrajSingh9886/Agriculture-Plan-Diseases-QA-Pairs-Dataset", split="train")
    datasets["CropDisease"] = ds
    print(f"CropDisease: {len(ds)} samples")
    print(f"   Columns: {ds.column_names}")
except Exception as e:
    print(f"CropDisease failed: {e}")

In [ ]:
# Format datasets for training - handles different column formats
def format_sample(sample):
    # Print column names for first sample (debug)
    if not hasattr(format_sample, 'printed'):
        format_sample.printed = True
        print(f"   Sample keys: {list(sample.keys())}")
    
    q = None
    a = None
    
    # Try various question column names
    for q_key in ['question', 'questions', 'QUESTION.question', 'input', 'instruction']:
        if q_key in sample and sample[q_key]:
            q = sample[q_key]
            break
    
    # Try various answer column names  
    for a_key in ['answer', 'answers', 'ANSWER', 'output', 'response', 'Response']:
        if a_key in sample and sample[a_key]:
            a = sample[a_key]
            break
    
    if q and a:
        return {"text": f"You are FarmGemma, an AI assistant for Indian farmers. Question: {q} Answer: {a}"}
    else:
        return {"text": ""}

print("\nFormatting datasets...")
all_datasets = []

for name, ds in datasets.items():
    print(f"\n{name}:")
    try:
        formatted = ds.map(format_sample)
        # Filter empty strings
        filtered = formatted.filter(lambda x: x['text'] != "")
        # Remove the text column temporarily to get just the text
        all_datasets.append(filtered)
        print(f"   Formatted: {len(filtered)} samples")
    except Exception as e:
        print(f"   Failed: {e}")

if all_datasets:
    train_dataset = concatenate_datasets(all_datasets)
    print(f"\nTOTAL TRAINING DATA: {len(train_dataset)} samples")
else:
    print("ERROR: No datasets loaded!")

In [ ]:
# Load Gemma 3 1B
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)

print("Loading Gemma 3 1B...")
model = AutoModelForCausalLM.from_pretrained("google/gemma-3-1b-it", quantization_config=quantization_config, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-1b-it")
print("Model loaded!")

In [ ]:
# Setup LoRA
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(r=8, lora_alpha=16, target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], lora_dropout=0.05, bias="none", task_type=TaskType.CAUSAL_LM)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Tokenize dataset
def tokenize_function(examples):
    result = tokenizer(examples["text"], truncation=True, max_length=512, padding="max_length")
    result["labels"] = result["input_ids"].copy()
    return result

print("Tokenizing...")
train_dataset = train_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
print(f"Tokenized: {len(train_dataset)} samples")

In [ ]:
# Training configuration
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir=f"{MODEL_DIR}/outputs",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=50,
    logging_steps=50,
    save_steps=500,
    save_total_limit=2,
    fp16=True,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    max_grad_norm=1.0,
    report_to="none"
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(model=model, args=training_args, train_dataset=train_dataset, data_collator=data_collator)
print(f"Ready to train {len(train_dataset)} samples!")

In [ ]:
# START TRAINING
print("Starting training... (1.5-2 hours with 3 epochs)")
trainer.train()

In [ ]:
# Save model
model.save_pretrained(f"{MODEL_DIR}/farmgemma-1b-agri")
tokenizer.save_pretrained(f"{MODEL_DIR}/farmgemma-1b-agri")
print(f"Model saved to {MODEL_DIR}/farmgemma-1b-agri")

In [ ]:
# Test the model
test_q = "How to control pink bollworm in cotton?"
prompt = f"You are FarmGemma. Question: {test_q} Answer:"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=200)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"Q: {test_q}")
print(f"A: {response.split('Answer:')[-1].strip()}")